Requirement
 1. Read raw data from flight_time_raw table
 2. Apply transformations to time values as hour to minute interval
     1. CRS_DEP_TIME
     2. DEP_TIME
     3. WHEELS_ON
     4. CRS_ARR_TIME
     5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

Read data using SQL Query.

In [0]:
%sql
select CRS_DEP_TIME,DEP_TIME,WHEELS_ON,CRS_ARR_TIME,ARR_TIME from dev_catalog.spark_db.flight_time_raw

1. Read data to create a dataframe.

In [0]:
flight_time_raw_df = spark.read.table("dev_catalog.spark_db.flight_time_raw")

2. Develop logic to transform CRS_DEP_TIME to an interval

In [0]:
from pyspark.sql.functions import expr
df_1 = ( flight_time_raw_df
.withColumn("CRS_DEP_TIME_HH", expr("left(lpad(CRS_DEP_TIME,4,'0'),2)"))
.withColumn("CRS_DEP_TIME_MM",expr("right(lpad(CRS_DEP_TIME,4,'0'),2)"))
)

df_2 = (df_1
.withColumn("CRS_DEP_TIME_NEW", expr("cast(concat(CRS_DEP_TIME_HH,':', CRS_DEP_TIME_MM)as INTERVAL HOUR TO MINUTE)"))
)
df_2.limit(5).display()



3. Develop a resuable function.

In [0]:
def get_interval(hhmm_value):
    from pyspark.sql.functions import expr

    return expr(f"""
                cast(concat(left(lpad({hhmm_value},4,'0'),2),':', right(lpad({hhmm_value},4,'0'),2)) AS INTERVAL HOUR TO MINUTE)""")

3. Apply function to dataframe

In [0]:
result_df = (
    flight_time_raw_df
    .withColumn("CRS_DEP_TIME",get_interval("CRS_DEP_TIME"))
    .withColumn("DEP_TIME",get_interval("DEP_TIME"))
)
type(result_df)

In [0]:
result_df_2 = (
    result_df.withColumns({
        "WHEELS_ON" : get_interval("WHEELS_ON"),
        "CRS_ARR_TIME" : get_interval("CRS_ARR_TIME"),
        "ARR_TIME" : get_interval("ARR_TIME"),
        "TAXI_IN" : get_interval("TAXI_IN")})
).display()

In [0]:
result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
        "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

5. Save results to the table 

In [0]:
result_df.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.flight_time")

In [0]:

%sql

select * from dev_catalog.spark_db.flight_time